# 2. Resultados experimentais

Lê os resultados produzidos por `python -m fraude experiment`, `train-final`, `explain` e `drift`.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40)
DATASET = "creditcard"   # ou "paysim"
REPORTS = ROOT / "reports" / DATASET

In [ ]:
exp = REPORTS / 'experiments'
info = json.loads((exp / 'dataset.json').read_text(encoding='utf-8'))
info

## Tabela comparativa

Métricas no conjunto de teste, com limiar escolhido pela função de custo (cenário moderado, FN:FP = 20:1) nas predições *out-of-fold* do treino.

In [ ]:
table = pd.read_csv(exp / 'results.csv')
table

In [ ]:
display(Markdown((exp / 'summary.md').read_text(encoding='utf-8')))

## AUC-PR por algoritmo × balanceamento

In [ ]:
display(Image(exp / 'figures' / 'auc_pr_heatmap.png'))

## Curvas precisão-revocação e ROC

A curva ROC tende a parecer otimista em bases desbalanceadas (Saito; Rehmsmeier, 2015).

In [ ]:
display(Image(exp / 'figures' / 'pr_curves.png'))
display(Image(exp / 'figures' / 'roc_curves.png'))

## Função de custo e escolha do limiar

In [ ]:
for s in ['conservador', 'moderado', 'agressivo']:
    display(Image(exp / 'figures' / f'cost_curves_{s}.png'))
display(Image(exp / 'figures' / 'confusion_moderado.png'))

## Custo computacional

In [ ]:
display(Image(exp / 'figures' / 'tradeoff_latency_aucpr.png'))
table[['model','balancing','train_seconds','latency_mean_ms','latency_p95_ms']].sort_values('latency_mean_ms')

## Duplicatas

Transações repetidas são removidas antes da separação treino/teste. A pasta `reports/creditcard_v1_com_duplicatas` guarda a execução anterior, para comparação.

In [ ]:
info.get('duplicates_removed'), info.get('duplicate_frauds_removed')

## Busca ampliada e ensembles

In [ ]:
for d in ['experiments_busca50', 'experiments_ensemble']:
    p = REPORTS / d / 'results.csv'
    if p.exists():
        display(Markdown(f'### {d}')); display(pd.read_csv(p))

## Comparação estatística

CV repetida 5×3; Friedman + Nemenyi; teste t corrigido de Nadeau-Bengio contra a melhor (Holm). Escolha final: a configuração mais rápida entre as não significativamente piores que a melhor.

In [ ]:
cmp = REPORTS / 'comparacao'
display(Markdown((cmp / 'summary.md').read_text(encoding='utf-8')))
display(Image(cmp / 'ranking_cd.png'))
json.loads((cmp / 'selection.json').read_text(encoding='utf-8'))

## Calibração do modelo final

In [ ]:
cal = REPORTS / 'calibracao'
display(pd.DataFrame({k: v for k, v in json.loads((cal / 'calibration.json').read_text(encoding='utf-8')).items() if k.startswith('test')}))
display(Image(cal / 'reliability.png'))

## Explicabilidade (SHAP e LIME)

In [ ]:
ex = REPORTS / 'explainability'
summary = json.loads((ex / 'explainability.json').read_text(encoding='utf-8'))
display(pd.DataFrame(summary['global_top10']))
display(Image(ex / 'shap_beeswarm.png'))
for case in summary['cases']:
    print(case, summary['cases'][case]['probability'], 'sobreposição SHAP×LIME top5:', summary['cases'][case]['top5_overlap'])
    display(Image(ex / f'shap_waterfall_{case}.png'))
    display(Image(ex / f'lime_{case}.png'))

## Concept drift

In [ ]:
dr = REPORTS / 'drift'
drift = pd.read_csv(dr / 'drift.csv')
display(drift.pivot(index='window', columns='strategy', values='auc_pr'))
display(drift[drift.strategy=='estatico'][['window','n_fraud','psi_score','psi_amount']])
display(Image(dr / 'drift.png'))